# Proyecto Final - Regresión con MLOps
**Autor:** Alex  
**Dataset:** California Housing Prices  
**Objetivo:** Predecir el valor medio de casas en California

## 1. Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente')

## 2. Carga de Datos

In [ ]:
# Cargar dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame

print(f'Shape del dataset: {df.shape}')
print(f'\nColumnas: {list(df.columns)}')
df.head()

## 3. Análisis Exploratorio (EDA)

In [ ]:
# Estadísticas descriptivas
df.describe()

In [ ]:
# Valores nulos
print('Valores nulos por columna:')
print(df.isnull().sum())

In [ ]:
# Distribución de la variable objetivo
plt.figure(figsize=(8, 4))
sns.histplot(df['MedHouseVal'], bins=50, kde=True)
plt.title('Distribución del Valor Medio de Casas')
plt.xlabel('Valor Medio (en $100,000)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Matriz de Correlación')
plt.tight_layout()
plt.show()

## 4. Preparación de Datos

In [ ]:
# Separar features y target
X = df.drop(columns=['MedHouseVal'])
y = df['MedHouseVal']

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

In [ ]:
# Escalado de features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('Escalado completado')

## 5. Entrenamiento del Modelo

In [ ]:
# Modelo 1: Regresión Lineal
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
print('Regresión Lineal entrenada')

In [ ]:
# Modelo 2: Random Forest Regressor
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
print('Random Forest entrenado')

## 6. Evaluación del Modelo

In [ ]:
def evaluar_modelo(nombre, modelo, X_test, y_test):
    y_pred = modelo.predict(X_test)
    mse  = mean_squared_error(y_test, y_pred)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    rmse = np.sqrt(mse)
    print(f'--- {nombre} ---')
    print(f'  RMSE : {rmse:.4f}')
    print(f'  MAE  : {mae:.4f}')
    print(f'  R2   : {r2:.4f}')
    return y_pred

y_pred_lr = evaluar_modelo('Regresión Lineal', lr, X_test_scaled, y_test)
print()
y_pred_rf = evaluar_modelo('Random Forest',    rf, X_test_scaled, y_test)

In [ ]:
# Gráfico: Predicciones vs Valores reales
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, titulo in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Regresión Lineal', 'Random Forest']
):
    ax.scatter(y_test, y_pred, alpha=0.3, s=10)
    ax.plot([y_test.min(), y_test.max()],
            [y_test.min(), y_test.max()], 'r--', lw=2)
    ax.set_xlabel('Valores Reales')
    ax.set_ylabel('Predicciones')
    ax.set_title(titulo)

plt.tight_layout()
plt.show()

## 7. Conclusiones

- Se entrenaron dos modelos de regresión sobre el dataset California Housing.
- **Random Forest** supera a la Regresión Lineal en todas las métricas.
- El R² del Random Forest indica que el modelo explica una alta proporción de la varianza.
- Próximos pasos: optimización de hiperparámetros, integración con MLflow y pipelines con Airflow.